# Herramientas · Pruebas basadas en propiedades con Hypothesis

Una prueba con ejemplos comprueba unos pocos casos elegidos a mano; una prueba basada en propiedades enuncia algo que debe cumplirse **para cualquier entrada** y deja que [Hypothesis](https://hypothesis.readthedocs.io/) genere cientos de entradas, incluidos los casos de borde (listas vacías, cero, negativos, NaN…). Cuando encuentra un fallo, lo reduce (*shrinking*) hasta el contraejemplo más simple. Aquí se practican las estrategias, las propiedades típicas (ida y vuelta, equivalencia con una versión ingenua, leyes algebraicas), cómo construir buenas estrategias, `@example`, `settings` y las pruebas con estado.

Las pruebas se ejecutan llamando directamente a las funciones decoradas, como haría `pytest`. Para que las salidas sean reproducibles se carga un perfil con `derandomize=True` (los ejemplos salen de una semilla fija derivada de la prueba) y sin base de datos de fallos.

In [1]:
import math
from collections import Counter
from itertools import chain

from hypothesis import HealthCheck, assume, example, find, given, seed, settings
from hypothesis import strategies as st

settings.register_profile("cuaderno", derandomize=True, database=None, deadline=None)
settings.load_profile("cuaderno")


def ejecutar(prueba):
    """Ejecuta una prueba; si falla, muestra el contraejemplo (Hypothesis lo deja en las notas) y devuelve la excepción."""
    try:
        prueba()
    except Exception as e:
        print(f"{prueba.__name__}: FALLA ({type(e).__name__})")
        if not hasattr(e, "__notes__"):
            print("   ", str(e).splitlines()[0])
        for nota in getattr(e, "__notes__", []):
            # La explicación que sigue incluye rutas de ficheros locales
            if nota.startswith("Explanation"):
                break
            print("   ", nota.replace("\n", "\n    "))
        return e
    print(f"{prueba.__name__}: pasa")


def contraejemplo(error):
    return "\n".join(getattr(error, "__notes__", []))

## 1. De ejemplos a una propiedad

Escribir `maximo(xs)` con `functools.reduce` y probarla primero con tres ejemplos. Después enunciar la propiedad general con `@given(st.lists(st.integers(), min_size=1))`: el resultado está en la lista y ningún elemento lo supera. Contar cuántas listas genera Hypothesis y ver qué contraejemplo encuentra en una versión que empieza a comparar desde 0.

In [2]:
from functools import reduce


def maximo(xs):
    return reduce(lambda a, b: a if a >= b else b, xs)


def maximo_desde_cero(xs):
    return reduce(lambda a, b: a if a >= b else b, xs, 0)


assert maximo([3, 9, 2]) == 9 and maximo([5]) == 5 and maximo([-2, -7]) == -2

generadas = 0


@given(st.lists(st.integers(), min_size=1))
def prueba_maximo(xs):
    global generadas
    generadas += 1
    m = maximo(xs)
    assert m in xs and all(x <= m for x in xs)


@given(st.lists(st.integers(), min_size=1))
def prueba_maximo_desde_cero(xs):
    m = maximo_desde_cero(xs)
    assert m in xs and all(x <= m for x in xs)


assert ejecutar(prueba_maximo) is None
print("Listas generadas:", generadas)
error = ejecutar(prueba_maximo_desde_cero)
assert "xs=[-1]" in contraejemplo(error)

prueba_maximo: pasa
Listas generadas: 100
prueba_maximo_desde_cero: FALLA (AssertionError)
    Failing test case: prueba_maximo_desde_cero(
        xs=[-1],
    )


## 2. Estrategias y el caso de NaN

Escribir `muestras(estrategia, n)`, que devuelva los valores que genera una estrategia dentro de una prueba (en lugar de `.example()`, pensado solo para explorar en la consola), y mostrar algunos de `st.integers`, `st.text`, `st.floats`, `st.tuples` y `st.dictionaries`. Después comprobar la propiedad `x == x` con `st.floats()`: falla con NaN, que no es igual a sí mismo, y pasa con `allow_nan=False`.

In [3]:
def muestras(estrategia, n=40):
    vistos = []

    @settings(max_examples=n)
    @given(estrategia)
    def recoger(x):
        vistos.append(x)

    recoger()
    return vistos


estrategias = {
    "integers(-100, 100)": st.integers(-100, 100),
    "text('aeiou', max_size=4)": st.text(alphabet="aeiou", max_size=4),
    "floats(0, 1)": st.floats(0, 1),
    "tuples(hora, minuto)": st.tuples(st.integers(0, 23), st.integers(0, 59)),
    "dictionaries(isla, plazas)": st.dictionaries(st.sampled_from(["TF", "GC", "LP"]), st.integers(0, 9), max_size=2),
}
for nombre, estrategia in estrategias.items():
    valores = muestras(estrategia)
    print(f"{nombre:<28} {valores[-4:]}")
    assert len(valores) > 1


@given(st.floats())
def prueba_igual_a_si_mismo(x):
    assert x == x


@given(st.floats(allow_nan=False))
def prueba_igual_a_si_mismo_sin_nan(x):
    assert x == x


assert "x=nan" in contraejemplo(ejecutar(prueba_igual_a_si_mismo))
assert ejecutar(prueba_igual_a_si_mismo_sin_nan) is None

integers(-100, 100)          [96, 32, 86, -47]
text('aeiou', max_size=4)    ['ui', 'i', 'uo', 'aouo']
floats(0, 1)                 [0.9459614254580575, 1.1754943508222875e-38, 0.5865520479247378, 0.8925802516261938]
tuples(hora, minuto)         [(0, 34), (20, 53), (4, 24), (0, 24)]
dictionaries(isla, plazas)   [{'LP': 7, 'TF': 7}, {'GC': 2, 'TF': 9}, {'GC': 9, 'TF': 9}, {'GC': 2}]
prueba_igual_a_si_mismo: FALLA (AssertionError)
    Failing test case: prueba_igual_a_si_mismo(
        x=nan,
    )
prueba_igual_a_si_mismo_sin_nan: pasa


## 3. Propiedades de ida y vuelta

Comprobar con propiedades que deshacer una transformación devuelve el original: comprimir y descomprimir un texto con RLE (`"AAAC"` → `[("A", 3), ("C", 1)]`), pasar una hora a minutos y volver, guardar un diccionario en JSON y leerlo, e invertir una lista dos veces. Comprobar también que ordenar es idempotente y conserva los elementos (mismo `Counter`).

In [4]:
import json
from itertools import groupby


def comprimir(texto):
    return [(c, len(list(g))) for c, g in groupby(texto)]


def descomprimir(pares):
    return "".join(c * n for c, n in pares)


def a_minutos(hora):
    h, m = hora
    return 60 * h + m


def a_hora(minutos):
    return divmod(minutos, 60)


@given(st.text(alphabet="ACGT"))
def prueba_rle(texto):
    pares = comprimir(texto)
    assert descomprimir(pares) == texto
    assert all(a != b for (a, _), (b, _) in zip(pares, pares[1:]))


@given(st.tuples(st.integers(0, 23), st.integers(0, 59)))
def prueba_hora(hora):
    assert a_hora(a_minutos(hora)) == hora


@given(st.dictionaries(st.text(), st.integers()))
def prueba_json(d):
    assert json.loads(json.dumps(d)) == d


@given(st.lists(st.integers()))
def prueba_invertir_y_ordenar(xs):
    assert xs[::-1][::-1] == xs
    assert sorted(sorted(xs)) == sorted(xs)
    assert Counter(sorted(xs)) == Counter(xs)


for prueba in (prueba_rle, prueba_hora, prueba_json, prueba_invertir_y_ordenar):
    assert ejecutar(prueba) is None
assert comprimir("AAAC") == [("A", 3), ("C", 1)]

prueba_rle: pasa


prueba_hora: pasa


prueba_json: pasa
prueba_invertir_y_ordenar: pasa


## 4. Implementación rápida frente a ingenua

Una forma muy eficaz de probar un algoritmo es compararlo con una versión lenta pero evidente. Comprobar que la búsqueda binaria encuentra un elemento en una lista ordenada exactamente cuando lo encuentra la búsqueda lineal (generando listas ordenadas con `.map(sorted)`), y que el máximo común divisor de Euclides coincide con `math.gcd` para cualquier par de enteros, también negativos o cero.

In [5]:
def busqueda_binaria(xs, x):
    izquierda, derecha = 0, len(xs) - 1
    while izquierda <= derecha:
        medio = (izquierda + derecha) // 2
        if xs[medio] == x:
            return medio
        if xs[medio] < x:
            izquierda = medio + 1
        else:
            derecha = medio - 1
    return -1


def busqueda_lineal(xs, x):
    return next((i for i, y in enumerate(xs) if y == x), -1)


def mcd(a, b):
    while b:
        a, b = b, a % b
    return abs(a)


@given(st.lists(st.integers(-20, 20)).map(sorted), st.integers(-25, 25))
def prueba_busqueda(xs, x):
    i, j = busqueda_binaria(xs, x), busqueda_lineal(xs, x)
    assert (i == -1) == (j == -1)
    if i != -1:
        assert xs[i] == xs[j] == x


@given(st.integers(), st.integers())
def prueba_mcd(a, b):
    assert mcd(a, b) == math.gcd(a, b)


assert ejecutar(prueba_busqueda) is None
assert ejecutar(prueba_mcd) is None

prueba_busqueda: pasa


prueba_mcd: pasa


## 5. Leyes algebraicas y redondeo

Comprobar que la suma de enteros es conmutativa, asociativa y tiene el 0 como neutro. Con `float` la conmutativa se cumple, pero la asociativa no: mostrar el contraejemplo mínimo y comprobar que con `Fraction` (aritmética exacta) sí se cumple. Usar también `find`, que devuelve directamente el ejemplo más simple que encuentra de una condición, para buscar un número cuyo cuadrado acabe en 444, y compararlo con el menor de todos obtenido por fuerza bruta: el *shrinking* da un ejemplo sencillo, pero no garantiza el mínimo.

In [6]:
from fractions import Fraction

finitos = st.floats(allow_nan=False, allow_infinity=False)


@given(st.integers(), st.integers(), st.integers())
def prueba_leyes_enteros(a, b, c):
    assert a + b == b + a
    assert (a + b) + c == a + (b + c)
    assert a + 0 == a


@given(finitos, finitos)
def prueba_conmutativa_float(a, b):
    assert a + b == b + a


@given(finitos, finitos, finitos)
def prueba_asociativa_float(a, b, c):
    assert (a + b) + c == a + (b + c)


@given(st.fractions(), st.fractions(), st.fractions())
def prueba_asociativa_fraction(a, b, c):
    assert (a + b) + c == a + (b + c)


for prueba in (prueba_leyes_enteros, prueba_conmutativa_float, prueba_asociativa_float, prueba_asociativa_fraction):
    error = ejecutar(prueba)
    assert (error is not None) == (prueba is prueba_asociativa_float)

acaba_en_444 = lambda n: str(n * n).endswith("444")
encontrado = find(st.integers(min_value=1), acaba_en_444)
menor = next(n for n in range(1, 10_000) if acaba_en_444(n))
print(f"\nfind: {encontrado}² = {encontrado**2}; el menor es {menor}² = {menor**2}")
assert acaba_en_444(encontrado) and encontrado >= menor

prueba_leyes_enteros: pasa
prueba_conmutativa_float: pasa


prueba_asociativa_float: FALLA (AssertionError)
    Failing test case: prueba_asociativa_float(
        a=1.0,
        b=1.0,
        c=9007199254740992.0,
    )
prueba_asociativa_fraction: pasa

find: 4462² = 19909444; el menor es 38² = 1444


## 6. `assume` y `filter` frente a construir la estrategia

Para probar que `(a // b) * b + a % b == a` hay que excluir `b = 0`. Hacerlo de tres formas: con `assume`, con `.filter` y construyendo directamente los divisores válidos. Ver que un filtro que descarta casi todo (múltiplos de 1000) hace fallar la comprobación de salud `filter_too_much`, y sustituirlo por `.map`. Por último, generar objetos con `st.builds` (una reserva de alojamiento) y datos relacionados con `@st.composite` (una lista ordenada junto con uno de sus elementos).

In [7]:
from dataclasses import dataclass


@given(st.integers(), st.integers())
def prueba_division_assume(a, b):
    assume(b != 0)
    assert (a // b) * b + a % b == a


@given(st.integers(), st.integers().filter(lambda b: b != 0))
def prueba_division_filter(a, b):
    assert (a // b) * b + a % b == a


divisores = st.one_of(st.integers(min_value=1), st.integers(max_value=-1))


@given(st.integers(), divisores)
def prueba_division_construida(a, b):
    assert (a // b) * b + a % b == a


@given(st.integers(1, 10**6).filter(lambda n: n % 1000 == 0))
def prueba_miles_filtrados(n):
    assert str(n).endswith("000")


@given(st.integers(1, 1000).map(lambda k: 1000 * k))
def prueba_miles_construidos(n):
    assert str(n).endswith("000")


for prueba in (prueba_division_assume, prueba_division_filter, prueba_division_construida):
    assert ejecutar(prueba) is None
assert type(ejecutar(prueba_miles_filtrados)).__name__ == "FailedHealthCheck"
assert ejecutar(prueba_miles_construidos) is None

prueba_division_assume: pasa
prueba_division_filter: pasa
prueba_division_construida: pasa
prueba_miles_filtrados: FALLA (FailedHealthCheck)
    It looks like this test is filtering out a lot of inputs. 1 inputs were generated successfully, while 50 inputs were filtered out. 
prueba_miles_construidos: pasa


In [8]:
@dataclass
class Reserva:
    isla: str
    noches: int
    personas: int

    def precio(self, tarifa=45, tasa=1.5):
        return self.noches * (tarifa + tasa * self.personas)


reservas = st.builds(Reserva, isla=st.sampled_from(["Tenerife", "La Gomera", "Lanzarote"]),
                     noches=st.integers(1, 30), personas=st.integers(1, 6))


@given(reservas)
def prueba_precio_reserva(r):
    assert r.precio() > 0
    assert Reserva(r.isla, r.noches + 1, r.personas).precio() > r.precio()


@st.composite
def lista_y_elemento(draw):
    xs = sorted(draw(st.lists(st.integers(-50, 50), min_size=1)))
    i = draw(st.integers(0, len(xs) - 1))
    return xs, xs[i]


@given(lista_y_elemento())
def prueba_binaria_encuentra(par):
    xs, x = par
    assert xs[busqueda_binaria(xs, x)] == x


assert ejecutar(prueba_precio_reserva) is None
assert ejecutar(prueba_binaria_encuentra) is None

prueba_precio_reserva: pasa
prueba_binaria_encuentra: pasa


## 7. Un fallo de borde y su reducción

`trocear(xs, n)` debe partir una lista en trozos de `n` elementos (el último puede ser más corto), pero tiene un error de borde a propósito: pierde el trozo final incompleto. Enunciar la propiedad (los trozos, unidos con `chain`, reconstruyen la lista y ninguno tiene más de `n` elementos), guardar cada entrada que falla y comparar la primera que encuentra Hypothesis con la última, ya reducida. Arreglar la función y comprobar que la propiedad pasa.

In [9]:
def trocear_con_error(xs, n):
    return [xs[i:i + n] for i in range(0, len(xs) - n + 1, n)]


def trocear(xs, n):
    return [xs[i:i + n] for i in range(0, len(xs), n)]


def prueba_trocear(funcion, fallos):
    @given(st.lists(st.integers()), st.integers(1, 10))
    def prueba(xs, n):
        trozos = funcion(xs, n)
        correcto = list(chain.from_iterable(trozos)) == xs and all(len(t) <= n for t in trozos)
        if not correcto:
            fallos.append((xs, n))
        assert correcto

    prueba.__name__ = f"prueba_{funcion.__name__}"
    return prueba


fallos = []
ejecutar(prueba_trocear(trocear_con_error, fallos))
print(f"\nEntradas que fallan: {len(fallos)}")
print("Primera encontrada:", fallos[0])
print("Reducida:          ", fallos[-1])
assert fallos[-1] == ([0], 2)
assert ejecutar(prueba_trocear(trocear, [])) is None

prueba_trocear_con_error: FALLA (AssertionError)
    Failing test case: prueba(
        xs=[0],
        n=2,
    )

Entradas que fallan: 14
Primera encontrada: ([-310097, 18528, 21919856938391, 5632334737099, -385], 2)
Reducida:           ([0], 2)
prueba_trocear: pasa


## 8. `@example`: fijar casos concretos

`@example` añade entradas que se prueban **siempre**, antes que las generadas: casos de borde conocidos o fallos que ya se corrigieron. Comprobar `es_bisiesto(año)` contra `calendar.isleap` fijando 1900, 2000 y 2024, y ver que una versión que olvida la regla de los 400 años falla en el ejemplo explícito 2000 sin depender del azar.

In [10]:
import calendar


def es_bisiesto(año):
    return año % 4 == 0 and (año % 100 != 0 or año % 400 == 0)


def es_bisiesto_sin_400(año):
    return año % 4 == 0 and año % 100 != 0


def prueba_bisiesto(funcion):
    @example(1900)
    @example(2000)
    @example(2024)
    @given(st.integers(1, 9999))
    def prueba(año):
        assert funcion(año) == calendar.isleap(año)

    prueba.__name__ = f"prueba_{funcion.__name__}"
    return prueba


assert ejecutar(prueba_bisiesto(es_bisiesto)) is None
assert "año=2000" in contraejemplo(ejecutar(prueba_bisiesto(es_bisiesto_sin_400)))

prueba_es_bisiesto: pasa
prueba_es_bisiesto_sin_400: FALLA (AssertionError)
    Failing explicit example: prueba(
        año=2000,
    )


## 9. `settings`, `deadline` y `@seed`

Con `settings` se ajusta cada prueba: comprobar cuántos ejemplos se ejecutan con `max_examples` de 10 y de 300, y que con `st.booleans()` Hypothesis se detiene en cuanto agota los dos valores posibles. Ver que una prueba lenta supera el `deadline` (tiempo máximo por ejemplo) y falla con `DeadlineExceeded`. Por último, sin `derandomize`, fijar la semilla con `@seed` y comprobar que dos ejecuciones generan exactamente los mismos valores.

In [11]:
import time


def cuenta_ejemplos(estrategia, maximo):
    contador = []

    @settings(max_examples=maximo)
    @given(estrategia)
    def prueba(x):
        contador.append(x)

    prueba()
    return len(contador)


print("integers, max_examples=10: ", cuenta_ejemplos(st.integers(), 10))
print("integers, max_examples=300:", cuenta_ejemplos(st.integers(), 300))
print("booleans, max_examples=300:", cuenta_ejemplos(st.booleans(), 300))
assert cuenta_ejemplos(st.booleans(), 300) == 2


@settings(deadline=100, max_examples=3)
@given(st.integers())
def prueba_lenta(x):
    time.sleep(0.25)


assert type(ejecutar(prueba_lenta)).__name__ == "DeadlineExceeded"


def valores_con_semilla():
    vistos = []

    @seed(2026)
    @settings(derandomize=False, max_examples=20)
    @given(st.lists(st.integers(0, 99), max_size=3))
    def prueba(xs):
        vistos.append(xs)

    prueba()
    return vistos


primera, segunda = valores_con_semilla(), valores_con_semilla()
print("\nMismos valores con @seed(2026):", primera == segunda)
assert primera == segunda

integers, max_examples=10:  10


integers, max_examples=300: 300
booleans, max_examples=300: 2


prueba_lenta: FALLA (DeadlineExceeded)
    Failing test case: prueba_lenta(
        x=0,  # or any other generated value
    )

Mismos valores con @seed(2026): True


## 10. Prueba con estado: una pila

Una `RuleBasedStateMachine` genera **secuencias de operaciones** en lugar de valores sueltos. Crear una clase `Pila` (con `apilar`, `desapilar`, `cima` y `len`) y una máquina que aplique operaciones al azar a la pila y a una lista que hace de modelo, con `@precondition` para no desapilar una pila vacía y un `@invariant` que compruebe que ambas tienen la misma longitud.

In [12]:
from hypothesis.stateful import RuleBasedStateMachine, invariant, precondition, rule, run_state_machine_as_test


class Pila:
    def __init__(self):
        self._datos = []

    def apilar(self, x):
        self._datos.append(x)

    def desapilar(self):
        return self._datos.pop()

    def cima(self):
        return self._datos[-1]

    def __len__(self):
        return len(self._datos)


class MaquinaPila(RuleBasedStateMachine):
    def __init__(self):
        super().__init__()
        self.pila, self.modelo = Pila(), []

    @rule(x=st.integers())
    def apilar(self, x):
        self.pila.apilar(x)
        self.modelo.append(x)

    @precondition(lambda self: self.modelo)
    @rule()
    def desapilar(self):
        assert self.pila.desapilar() == self.modelo.pop()

    @precondition(lambda self: self.modelo)
    @rule()
    def cima(self):
        assert self.pila.cima() == self.modelo[-1]

    @invariant()
    def misma_longitud(self):
        assert len(self.pila) == len(self.modelo)


def prueba_pila():
    run_state_machine_as_test(MaquinaPila)


assert ejecutar(prueba_pila) is None

prueba_pila: pasa


## 11. Prueba con estado: una cola con un fallo

Una cola puede implementarse con dos pilas: se mete en `entrada` y se saca de `salida`, y cuando `salida` se vacía se vuelca en ella `entrada` invertida. Escribir una versión que olvida vaciar `entrada` al volcarla, probarla con una máquina de estados contra un `deque` y mostrar la secuencia mínima de operaciones que la rompe. Corregirla y comprobar que la máquina pasa.

In [13]:
from collections import deque


class ColaDosPilas:
    def __init__(self, vaciar_al_volcar=True):
        self.entrada, self.salida = [], []
        self.vaciar_al_volcar = vaciar_al_volcar

    def poner(self, x):
        self.entrada.append(x)

    def sacar(self):
        if not self.salida:
            self.salida = self.entrada[::-1]
            if self.vaciar_al_volcar:
                self.entrada = []
        return self.salida.pop()

    def __len__(self):
        return len(self.entrada) + len(self.salida)


def maquina_cola(vaciar_al_volcar):
    class MaquinaCola(RuleBasedStateMachine):
        def __init__(self):
            super().__init__()
            self.cola, self.modelo = ColaDosPilas(vaciar_al_volcar), deque()

        @rule(x=st.integers(0, 9))
        def poner(self, x):
            self.cola.poner(x)
            self.modelo.append(x)

        @precondition(lambda self: self.modelo)
        @rule()
        def sacar(self):
            assert self.cola.sacar() == self.modelo.popleft()

        @invariant()
        def misma_longitud(self):
            assert len(self.cola) == len(self.modelo)

    return MaquinaCola


def prueba_cola_con_error():
    run_state_machine_as_test(maquina_cola(vaciar_al_volcar=False))


def prueba_cola():
    run_state_machine_as_test(maquina_cola(vaciar_al_volcar=True))


assert "state.sacar()" in contraejemplo(ejecutar(prueba_cola_con_error))
assert ejecutar(prueba_cola) is None

prueba_cola_con_error: FALLA (AssertionError)
    Failing test case: <state machine>
    state = MaquinaCola()
    state.misma_longitud()
    state.poner(x=0)
    state.misma_longitud()
    state.sacar()
    state.misma_longitud()
    state.teardown()


prueba_cola: pasa
